# Genie Reasoning Process Logger

Captures the full reasoning process of Genie space conversations — each step logged as a separate row with timestamps and content.

| Step Type | What it captures | Source |
| --- | --- | --- |
| **reasoning** | AI description, understanding, data sourcing, and planned steps | `attachments[].query.thoughts[]` |
| **function_call** | Generated SQL query | `attachments[].query.query` |
| **function_call_output** | Query result metadata (row count, statement ID) | `attachments[].query.query_result_metadata` |
| **message** | Final text answer / report | `attachments[].text.content` |

### Parameters

| Widget | Purpose |
| --- | --- |
| `space_id` | Genie space to pull conversations from |
| `catalog_name` | Target Unity Catalog catalog |
| `schema_name` | Target schema |
| `table_name` | Target table (default: `genie_reasoning_steps`) |

In [ ]:
# Parameters
dbutils.widgets.text("space_id", "", "Genie Space ID")
dbutils.widgets.text("catalog_name", "", "Target Catalog")
dbutils.widgets.text("schema_name", "", "Target Schema")
dbutils.widgets.text("table_name", "genie_reasoning_steps", "Target Table Name")

SPACE_ID = dbutils.widgets.get("space_id")
CATALOG = dbutils.widgets.get("catalog_name")
SCHEMA = dbutils.widgets.get("schema_name")
TABLE_NAME = dbutils.widgets.get("table_name")
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(f"Space ID: {SPACE_ID}")
print(f"Target table: {FULL_TABLE_NAME}")

In [ ]:
import requests
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

# Workspace context for API calls
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = ctx.apiUrl().get()
TOKEN = ctx.apiToken().get()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


def api_get(path: str, params: dict | None = None) -> dict:
    """GET request to the Databricks REST API with retry on 429."""
    url = f"{HOST}{path}"
    for attempt in range(3):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if resp.status_code == 429:
            import time
            time.sleep(2 ** attempt)
            continue
        resp.raise_for_status()
        return resp.json()
    resp.raise_for_status()
    return resp.json()


print(f"Connected to: {HOST}")

In [ ]:
def list_conversations(space_id: str) -> list[dict]:
    """Retrieve all conversations for a Genie space with pagination."""
    conversations, page_token = [], None
    while True:
        params = {"page_size": 100}
        if page_token:
            params["page_token"] = page_token
        data = api_get(f"/api/2.0/genie/spaces/{space_id}/conversations", params=params)
        convs = data.get("conversations", [])
        conversations.extend(convs)
        page_token = data.get("next_page_token")
        if not page_token or not convs:
            break
    return conversations


def list_messages(space_id: str, conversation_id: str) -> list[dict]:
    """Retrieve all messages for a given conversation with pagination."""
    messages, page_token = [], None
    while True:
        params = {"page_size": 100}
        if page_token:
            params["page_token"] = page_token
        data = api_get(
            f"/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages",
            params=params,
        )
        msgs = data.get("messages", [])
        messages.extend(msgs)
        page_token = data.get("next_page_token")
        if not page_token or not msgs:
            break
    return messages


conversations = list_conversations(SPACE_ID)
print(f"Found {len(conversations)} conversations")

In [ ]:
import time as _time

# ── Phase-to-step mapping ─────────────────────────────────────────────
# Status transitions observed during polling:
#   SUBMITTED → FILTERING_CONTEXT → ASKING_AI → PENDING_WAREHOUSE → ASKING_AI → COMPLETED
# We map each transition to a reasoning step type:
_PHASE_TO_STEP = {
    "FILTERING_CONTEXT": "reasoning",    # context filtering / understanding
    "ASKING_AI":         "function_call", # first pass = SQL generation; later = answer gen
    "PENDING_WAREHOUSE": "function_call_output",  # SQL executing on warehouse
    "COMPLETED":         "message",       # final answer ready
}


def _poll_until_complete(space_id: str, conv_id: str, msg_id: str,
                         max_polls: int = 120, interval: float = 1.5
                         ) -> tuple[dict, list[dict]]:
    """Poll a message until terminal status, capturing status transitions.

    Returns:
        (final_message_dict, transitions_list)
        Each transition = {"status": str, "timestamp_ms": int}
    """
    transitions = []
    prev_status = None
    final_msg = {}

    for _ in range(max_polls):
        data = api_get(
            f"/api/2.0/genie/spaces/{space_id}/conversations/{conv_id}/messages/{msg_id}"
        )
        status = data.get("status", "")
        ts = data.get("last_updated_timestamp") or data.get("created_timestamp", 0)

        if status != prev_status:
            transitions.append({"status": status, "timestamp_ms": ts})
            prev_status = status

        if status in ("COMPLETED", "FAILED", "CANCELLED"):
            final_msg = data
            break

        _time.sleep(interval)

    return final_msg, transitions


def _extract_reasoning_steps(conv: dict) -> list[dict]:
    """Extract reasoning steps from all messages in a conversation.

    For messages still in progress → polls actively to capture per-phase
    timestamps from status transitions.

    For completed messages → reads final state; per-phase timing is derived
    from system.query.history where available.
    """
    conversation_id = conv["conversation_id"]
    conv_title = conv.get("title", "")
    records = []

    try:
        messages = list_messages(SPACE_ID, conversation_id)
    except requests.HTTPError:
        return records

    for msg in messages:
        message_id = msg.get("message_id", "")
        status = msg.get("status", "")
        transitions = []

        # If message is still processing → poll to capture phase timestamps
        if status not in ("COMPLETED", "FAILED", "CANCELLED"):
            msg, transitions = _poll_until_complete(
                SPACE_ID, conversation_id, message_id
            )
            status = msg.get("status", status)

        created_ts = msg.get("created_timestamp", 0)
        updated_ts = msg.get("last_updated_timestamp", 0)
        user_question = (msg.get("content") or "")[:500] or None
        step_index = 0

        # Build per-phase timing from transitions (if captured)
        phase_timing = {}  # step_type → {"created_at_ms": int, "duration_ms": int}
        if transitions:
            # Add initial SUBMITTED marker at created_ts
            full_chain = [{"status": "SUBMITTED", "timestamp_ms": created_ts}] + transitions
            # Track which ASKING_AI occurrence we're on
            asking_ai_count = 0
            for i in range(1, len(full_chain)):
                phase_status = full_chain[i]["status"]
                phase_start = full_chain[i - 1]["timestamp_ms"]
                phase_end = full_chain[i]["timestamp_ms"]
                dur = phase_end - phase_start

                if phase_status == "ASKING_AI":
                    asking_ai_count += 1
                    if asking_ai_count == 1:
                        # First ASKING_AI = reasoning + SQL generation
                        phase_timing["reasoning"] = {
                            "created_at_ms": phase_start, "duration_ms": dur,
                        }
                    else:
                        # Subsequent ASKING_AI = answer generation
                        phase_timing["message"] = {
                            "created_at_ms": phase_start, "duration_ms": dur,
                        }
                elif phase_status == "PENDING_WAREHOUSE":
                    phase_timing["function_call"] = {
                        "created_at_ms": phase_start, "duration_ms": dur,
                    }
                    phase_timing["function_call_output"] = {
                        "created_at_ms": phase_start, "duration_ms": dur,
                    }
                elif phase_status == "COMPLETED" and "message" not in phase_timing:
                    phase_timing["message"] = {
                        "created_at_ms": phase_start, "duration_ms": dur,
                    }
                elif phase_status == "FILTERING_CONTEXT":
                    # Context filtering happens before reasoning
                    phase_timing.setdefault("reasoning", {
                        "created_at_ms": phase_start, "duration_ms": dur,
                    })

        base = {
            "space_id": SPACE_ID,
            "conversation_id": conversation_id,
            "conversation_title": conv_title,
            "message_id": message_id,
            "user_id": msg.get("user_id"),
            "user_question": user_question,
            "status": status,
            "question_timestamp": created_ts,
            "response_timestamp": updated_ts if updated_ts > created_ts else None,
        }

        for att in (msg.get("attachments") or []):
            query_info = att.get("query", {})
            text_info = att.get("text", {})

            # --- reasoning: thoughts from the query attachment ---
            for thought in query_info.get("thoughts", []):
                thought_type = thought.get("thought_type", "")
                subtype = thought_type.replace("THOUGHT_TYPE_", "") if thought_type else None
                step_index += 1
                pt = phase_timing.get("reasoning", {})
                records.append({
                    **base,
                    "step_index": step_index,
                    "step_type": "reasoning",
                    "step_subtype": subtype,
                    "content": thought.get("content", ""),
                    "generated_sql": None,
                    "sql_statement_id": None,
                    "result_row_count": None,
                    "step_created_at_ms": pt.get("created_at_ms"),
                    "step_duration_ms": pt.get("duration_ms"),
                })

            # --- function_call: the generated SQL query ---
            if query_info.get("query"):
                step_index += 1
                pt = phase_timing.get("function_call", {})
                records.append({
                    **base,
                    "step_index": step_index,
                    "step_type": "function_call",
                    "step_subtype": "execute_sql",
                    "content": query_info.get("description", ""),
                    "generated_sql": query_info["query"][:4000],
                    "sql_statement_id": query_info.get("statement_id"),
                    "result_row_count": None,
                    "step_created_at_ms": pt.get("created_at_ms"),
                    "step_duration_ms": pt.get("duration_ms"),
                })

            # --- function_call_output: query result metadata ---
            result_meta = query_info.get("query_result_metadata", {})
            if result_meta.get("row_count") is not None or query_info.get("statement_id"):
                step_index += 1
                row_count = result_meta.get("row_count")
                pt = phase_timing.get("function_call_output", {})
                records.append({
                    **base,
                    "step_index": step_index,
                    "step_type": "function_call_output",
                    "step_subtype": "query_result",
                    "content": f"Query returned {row_count} rows" if row_count is not None else "Query executed",
                    "generated_sql": None,
                    "sql_statement_id": query_info.get("statement_id"),
                    "result_row_count": row_count,
                    "step_created_at_ms": pt.get("created_at_ms"),
                    "step_duration_ms": pt.get("duration_ms"),
                })

            # --- message: final text answer ---
            if text_info.get("content") and text_info.get("purpose") == "TEXT_ATTACHMENT_PURPOSE_ANSWER":
                step_index += 1
                pt = phase_timing.get("message", {})
                records.append({
                    **base,
                    "step_index": step_index,
                    "step_type": "message",
                    "step_subtype": "answer",
                    "content": text_info["content"][:4000],
                    "generated_sql": None,
                    "sql_statement_id": None,
                    "result_row_count": None,
                    "step_created_at_ms": pt.get("created_at_ms"),
                    "step_duration_ms": pt.get("duration_ms"),
                })

    return records


# Parallel fetch: 8 workers (completed msgs are fast; in-progress msgs will poll)
print(f"Extracting reasoning steps from {len(conversations)} conversations...")
reasoning_records = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(_extract_reasoning_steps, c): c for c in conversations}
    for future in as_completed(futures):
        reasoning_records.extend(future.result())

# Summary by step type
from collections import Counter
type_counts = Counter(r["step_type"] for r in reasoning_records)
polled_count = sum(1 for r in reasoning_records if r.get("step_created_at_ms"))
print(f"Extracted {len(reasoning_records)} reasoning steps ({polled_count} with per-phase timing):")
for t, c in type_counts.most_common():
    print(f"  {t}: {c}")

In [ ]:
SCHEMA_DEF = StructType([
    StructField("space_id", StringType(), False),
    StructField("conversation_id", StringType(), False),
    StructField("conversation_title", StringType(), True),
    StructField("message_id", StringType(), False),
    StructField("user_id", LongType(), True),
    StructField("user_question", StringType(), True),
    StructField("status", StringType(), True),
    StructField("question_timestamp", LongType(), True),
    StructField("response_timestamp", LongType(), True),
    StructField("step_index", IntegerType(), True),
    StructField("step_type", StringType(), True),
    StructField("step_subtype", StringType(), True),
    StructField("content", StringType(), True),
    StructField("generated_sql", StringType(), True),
    StructField("sql_statement_id", StringType(), True),
    StructField("result_row_count", IntegerType(), True),
    StructField("step_created_at_ms", LongType(), True),
    StructField("step_duration_ms", LongType(), True),
])

if not reasoning_records:
    print("No reasoning records extracted.")
    df_new = spark.createDataFrame([], schema=SCHEMA_DEF)
else:
    df_base = spark.createDataFrame(reasoning_records, schema=SCHEMA_DEF)
    stmt_ids = list({r["sql_statement_id"] for r in reasoning_records if r["sql_statement_id"]})

    # --- Join query timing from system.query.history for function_call_output rows ---
    if stmt_ids:
        spark.createDataFrame([(s,) for s in stmt_ids], ["stmt_id"]).createOrReplaceTempView("_genie_reason_stmt_ids")
        df_qh = spark.sql("""
            SELECT statement_id,
                   BIGINT(UNIX_TIMESTAMP(start_time) * 1000) AS query_start_ms,
                   BIGINT(UNIX_TIMESTAMP(end_time) * 1000) AS query_end_ms,
                   execution_duration_ms
            FROM system.query.history
            INNER JOIN _genie_reason_stmt_ids ON statement_id = stmt_id
        """)
        df_base = (
            df_base.alias("b")
            .join(df_qh.alias("qh"), F.col("b.sql_statement_id") == F.col("qh.statement_id"), "left")
            .withColumn("query_start_ms", F.col("qh.query_start_ms"))
            .withColumn("query_end_ms", F.col("qh.query_end_ms"))
            .withColumn("execution_duration_ms", F.col("qh.execution_duration_ms"))
            .select("b.*", "query_start_ms", "query_end_ms", "execution_duration_ms")
        )
        # Propagate query timing to ALL steps within the same message
        # (reasoning/message rows don't have sql_statement_id but need the timestamps)
        from pyspark.sql.window import Window
        msg_window = Window.partitionBy("space_id", "conversation_id", "message_id")
        df_base = (
            df_base
            .withColumn("query_start_ms", F.max("query_start_ms").over(msg_window))
            .withColumn("query_end_ms", F.max("query_end_ms").over(msg_window))
        )
    else:
        df_base = (
            df_base
            .withColumn("query_start_ms", F.lit(None).cast("long"))
            .withColumn("query_end_ms", F.lit(None).cast("long"))
            .withColumn("execution_duration_ms", F.lit(None).cast("long"))
        )

    # --- Join conversation mode from audit log ---
    min_ts = min(r["question_timestamp"] for r in reasoning_records if r["question_timestamp"]) // 1000
    from datetime import date, timedelta
    earliest_date = date.fromtimestamp(min_ts) - timedelta(days=1)

    df_conv_mode = spark.sql(f"""
        SELECT
            get_json_object(response.result, '$.conversation_id') AS conversation_id,
            CASE request_params.conversation_type
                WHEN 'NORMAL' THEN 'chat'
                WHEN 'DEEP_RESEARCH' THEN 'agent'
            END AS conversation_mode
        FROM system.access.audit
        WHERE service_name = 'aibiGenie'
          AND action_name = 'createConversation'
          AND request_params.space_id = '{SPACE_ID}'
          AND request_params.conversation_type IS NOT NULL
          AND event_date >= '{earliest_date}'
    """)

    df_new = (
        df_base.alias("d")
        .join(df_conv_mode.alias("m"), F.col("d.conversation_id") == F.col("m.conversation_id"), "left")
        .select("d.*", F.coalesce(F.col("m.conversation_mode"), F.lit("chat")).alias("conversation_mode"))
        .withColumn("question_time", F.from_unixtime(F.col("question_timestamp") / 1000))
        .withColumn("response_time", F.from_unixtime(F.col("response_timestamp") / 1000))
        .withColumn("query_start_time", F.from_unixtime(F.col("query_start_ms") / 1000))
        .withColumn("query_end_time", F.from_unixtime(F.col("query_end_ms") / 1000))
        .withColumn("ingested_at", F.current_timestamp())
        # Total message duration (always available, never null)
        .withColumn(
            "total_duration_sec",
            F.round(
                (F.col("response_timestamp") - F.col("question_timestamp")) / 1000.0, 2
            ),
        )
        # --- Derive per-step timing from system.query.history as fallback ---
        # When active polling captured phase timing, use it (step_created_at_ms).
        # Otherwise, derive from query start/end times:
        #   reasoning steps:       created_at = question_timestamp, duration = query_start - question
        #   function_call:         created_at = query_start, duration = 0 (SQL was generated during reasoning)
        #   function_call_output:  created_at = query_start, duration = query_end - query_start
        #   message:               created_at = query_end, duration = response - query_end
        .withColumn(
            "step_created_at_ms",
            F.coalesce(
                F.col("step_created_at_ms"),  # from active polling
                F.when(F.col("step_type") == "reasoning", F.col("question_timestamp")),
                F.when(F.col("step_type") == "function_call", F.col("query_start_ms")),
                F.when(F.col("step_type") == "function_call_output", F.col("query_start_ms")),
                F.when(F.col("step_type") == "message", F.col("query_end_ms")),
            ),
        )
        .withColumn(
            "step_duration_ms",
            F.coalesce(
                F.col("step_duration_ms"),  # from active polling
                F.when(
                    F.col("step_type") == "reasoning",
                    F.col("query_start_ms") - F.col("question_timestamp"),
                ),
                F.when(F.col("step_type") == "function_call", F.lit(0)),
                F.when(
                    F.col("step_type") == "function_call_output",
                    F.col("query_end_ms") - F.col("query_start_ms"),
                ),
                F.when(
                    F.col("step_type") == "message",
                    F.col("response_timestamp") - F.col("query_end_ms"),
                ),
            ),
        )
        .withColumn("step_created_at", F.from_unixtime(F.col("step_created_at_ms") / 1000))
        .withColumn("step_duration_sec", F.round(F.col("step_duration_ms") / 1000.0, 2))
    )

    display(df_new.select(
        "user_question",
        "step_index", "step_type", "step_subtype",
        "content", "generated_sql",
        "step_created_at", "step_duration_sec",
        "total_duration_sec",
        F.round(F.col("execution_duration_ms") / 1000.0, 2).alias("query_exec_sec"),
        "question_time", "response_time",
        "conversation_mode", "result_row_count",
    ))

In [ ]:
# Ensure catalog and schema exist
spark.sql(f"USE CATALOG {CATALOG};")
spark.sql(f"USE SCHEMA {SCHEMA};")

# Create table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FULL_TABLE_NAME} (
    space_id STRING NOT NULL,
    conversation_id STRING NOT NULL,
    conversation_title STRING,
    message_id STRING NOT NULL,
    user_id BIGINT COMMENT 'Databricks user ID of the person who asked the question',
    user_question STRING,
    status STRING,
    step_index INT COMMENT 'Ordering of the step within the message (1-based)',
    step_type STRING NOT NULL COMMENT 'Type: reasoning, function_call, function_call_output, message',
    step_subtype STRING COMMENT 'Subtype: DESCRIPTION, UNDERSTANDING, DATA_SOURCING, STEPS, execute_sql, query_result, answer',
    content STRING COMMENT 'Text content of the reasoning step, SQL description, result summary, or final answer',
    generated_sql STRING COMMENT 'Generated SQL query (for function_call steps)',
    sql_statement_id STRING,
    result_row_count INT COMMENT 'Number of rows returned by the query (for function_call_output steps)',
    conversation_mode STRING COMMENT 'Genie mode: chat (NORMAL) or agent (DEEP_RESEARCH), from audit log',
    question_timestamp BIGINT,
    response_timestamp BIGINT,
    query_start_ms BIGINT COMMENT 'SQL query start time in ms (from system.query.history)',
    query_end_ms BIGINT COMMENT 'SQL query end time in ms (from system.query.history)',
    execution_duration_ms BIGINT COMMENT 'SQL execution duration in ms (from system.query.history)',
    question_time TIMESTAMP,
    response_time TIMESTAMP,
    query_start_time TIMESTAMP,
    query_end_time TIMESTAMP,
    ingested_at TIMESTAMP,
    total_duration_sec DOUBLE COMMENT 'Total message duration in seconds (response_timestamp - question_timestamp), always populated',
    step_created_at_ms BIGINT COMMENT 'Timestamp (ms) when this step phase started, from active polling',
    step_duration_ms BIGINT COMMENT 'Duration (ms) of this step phase, from active polling status transitions',
    step_created_at TIMESTAMP COMMENT 'When this step phase started (human-readable)',
    step_duration_sec DOUBLE COMMENT 'Duration of this step phase in seconds'
) USING DELTA
COMMENT 'Incremental log of Genie reasoning steps: each row is one step (reasoning, function_call, function_call_output, or message) with timestamps'
""")

print(f"Table {FULL_TABLE_NAME} ready.")

In [ ]:
from delta.tables import DeltaTable

if reasoning_records:
    target = DeltaTable.forName(spark, FULL_TABLE_NAME)

    (
        target.alias("t")
        .merge(
            df_new.alias("s"),
            """
            t.space_id = s.space_id
            AND t.conversation_id = s.conversation_id
            AND t.message_id = s.message_id
            AND t.step_type = s.step_type
            AND t.step_index = s.step_index
            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    row_count = spark.table(FULL_TABLE_NAME).count()
    print(f"Merge complete. Table now has {row_count} rows.")
else:
    print("No new records to merge.")

In [ ]:
# Overall summary
display(
    spark.sql(f"""
    SELECT
        space_id,
        COUNT(*) AS total_steps,
        COUNT(DISTINCT conversation_id) AS total_conversations,
        COUNT(DISTINCT message_id) AS total_messages,
        COUNT(DISTINCT user_id) AS distinct_users,
        ROUND(AVG(total_duration_sec), 2) AS avg_total_duration_sec,
        MIN(question_time) AS earliest_question,
        MAX(question_time) AS latest_question
    FROM {FULL_TABLE_NAME}
    GROUP BY space_id
    """)
)

# Breakdown by step type and subtype with per-phase timing
display(
    spark.sql(f"""
    SELECT
        step_type,
        step_subtype,
        COUNT(*) AS step_count,
        COUNT(DISTINCT message_id) AS messages_with_step,
        ROUND(AVG(step_duration_sec), 2) AS avg_step_duration_sec,
        ROUND(AVG(total_duration_sec), 2) AS avg_msg_duration_sec,
        ROUND(AVG(execution_duration_ms / 1000.0), 2) AS avg_query_exec_sec
    FROM {FULL_TABLE_NAME}
    GROUP BY step_type, step_subtype
    ORDER BY step_type, step_count DESC
    """)
)

# Show reasoning content and questions per conversation
display(
    spark.sql(f"""
    SELECT
        user_question,
        step_type,
        step_subtype,
        content,
        generated_sql,
        step_created_at,
        step_duration_sec,
        total_duration_sec,
        ROUND(execution_duration_ms / 1000.0, 2) AS query_exec_sec,
        result_row_count,
        question_time,
        response_time
    FROM {FULL_TABLE_NAME}
    ORDER BY conversation_id, message_id, step_index
    """)
)